In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "google/gemma-4-E4B-it" 
DEVICE = "cuda:3"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map=device,
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

In [3]:
model = load_model()

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [4]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [5]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [6]:
bridge


TransformerBridge(
  (vision_encoder): GeneralizedComponent(
    (hook_in): HookPoint(name='vision_encoder.hook_in')
    (hook_out): HookPoint(name='vision_encoder.hook_out')
    (_original_component): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              

In [7]:
model.model.language_model.layers[0].self_attn.is_kv_shared_layer

False

In [8]:
model.model.language_model.layers[0].self_attn.kv_shared_layer_index

In [9]:
model.model.language_model.layers[0].self_attn.store_full_length_kv

False

In [10]:
model.model.language_model.layers[0].self_attn.use_alternative_attention

False

In [11]:
model.model.language_model.layers[0].self_attn.o_proj._original_component.bias

In [12]:
model.model.language_model.layers[0].self_attn._original_component.config.attention_bias

False

In [13]:
model.model.language_model.layers[0].self_attn._original_component.config._attn_implementation

'sdpa'

In [14]:
model.model.language_model.layers[41].self_attn._original_component.config.use_double_wide_mlp

False

In [15]:
model.model.language_model.layers[0]._original_component.enable_moe_block
#TODO if we want to enable? If we want to prune it?

False

In [16]:
model.model.language_model.layers[0]._original_component.hidden_size_per_layer_input

256

основной поток 2560 ───────────────────────────────┐
                                                   + → 2560
основной поток 2560 → gate 256                     │
                              × per-layer input 256 │
                              → projection 2560 ────┘

There is another branch of per_layer_input in forward of model.

Builded from special per-layer embeddings. Or from projection of main embedding (???)

In [17]:
# token t:
    # main embedding/hidden state: 2560
    # layer 0 extra input:          256
    # layer 1 extra input:          256
    # ...
    # layer 41 extra input:         256

In [18]:
model.model.language_model.layers[0]._original_component.hidden_size

2560

In [19]:
model.model.language_model.layers[0]._original_component.config.hidden_activation

'gelu_pytorch_tanh'

In [20]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [21]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [22]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [23]:
ignored_params = []
# for name, param in model.named_parameters():
#     if "norm" in name:
#         ignored_params.append(param)

In [24]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input=input_ids,
        use_cache=False,
        return_type="logits",
        #return_dict=True,
    ) #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    bridge,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
        unwrapped_parameters=[(bridge.blocks[0].attn.q_norm._original_component.weight, 0),]
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.vision_tower._original_component.encoder.layers.9.post_attention_layernorm.weight', 'model.vision_tower._original_component.encoder.layers.13.self_attn.q_norm.weight', 'model.audio_tower.layers.6.feed_forward2.pre_layer_norm.weight', 'model.audio_tower.layers.6.lconv1d.linear_start.linear.weight', 'model.audio_tower.layers.9.feed_forward1.post_layer_norm.weight', 'model.language_model.layers.2._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.8._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.13._original_component.pre_feedforward_layernorm._original_component.weight', 'model.language_model.layers.22._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.28._original_compo

In [25]:
bridge.get_submodule('model.audio_tower.layers.1.self_attn.relative_k_proj')
#todo why unwrapped? because audio_tower?

Linear(in_features=1024, out_features=1024, bias=False)

In [26]:
model.model.audio_tower.layers[1].self_attn.relative_k_proj

Linear(in_features=1024, out_features=1024, bias=False)

In [27]:
bridge.blocks[0].attn.q

LinearBridge(2560 -> 2048, bias=False, original_component=Linear)

In [28]:
hasattr(bridge, 'rotary_emb')

True

In [29]:
bridge.get_submodule("blocks.0.attn.q.hook_out")

HookPoint(name='blocks.0.attn.q.hook_out')

In [30]:
#name of module, cols (in), rows(out)


#local configuration
d = {"blocks.0.attn.q": (None, [2, 6, 9]), #repeat indices for qkvo
     "blocks.0.mlp.up_proj": (None, [1, 3, 5])}

In [31]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )

In [32]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)), len(idxs)=3
[1] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on _Reshape_3747(), len(idxs)=3
[2] prune_out_channels on _Reshape_3747() => prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0) => prune_out_channels on _ElementWiseOp_3740(MulBackward0), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0) => prune_out_chann

In [33]:
def manually_indices_repeating(num_heads: int, head_dim: int, pruning_indices: torch.Tensor):
    all_indices = []
    for head_num in range(num_heads):
        all_indices.append(
            pruning_indices+head_num*head_dim)
    return torch.cat(all_indices)

In [34]:
type(bridge.blocks[0].attn)

transformer_lens.model_bridge.generalized_components.base.GeneralizedComponent

In [35]:
bridge

TransformerBridge(
  (vision_encoder): GeneralizedComponent(
    (hook_in): HookPoint(name='vision_encoder.hook_in')
    (hook_out): HookPoint(name='vision_encoder.hook_out')
    (_original_component): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              

In [36]:
#matches with config from llama
print(bridge.blocks[0].attn._original_component.config.num_attention_heads)
print(bridge.blocks[0].attn._original_component.config.head_dim)
print(bridge.blocks[0].attn._original_component.config.num_key_value_heads)

8
256
2


In [37]:
print(bridge.blocks[0].attn.q._original_component.weight.shape)
print(bridge.blocks[0].attn.k._original_component.weight.shape)


torch.Size([2048, 2560])
torch.Size([512, 2560])


In [38]:
bridge.blocks[0].attn._original_component.config.hidden_size

2560

In [39]:
bridge.blocks[0].attn.head_dim

256

In [40]:
repeated_idxs = manually_indices_repeating(
    bridge.blocks[0].attn._original_component.config.num_attention_heads,
    bridge.blocks[0].attn._original_component.config.head_dim,
    torch.tensor([2, 6, 9])
)
repeated_idxs

tensor([   2,    6,    9,  258,  262,  265,  514,  518,  521,  770,  774,  777,
        1026, 1030, 1033, 1282, 1286, 1289, 1538, 1542, 1545, 1794, 1798, 1801])

In [41]:
repeated_idxs.shape

torch.Size([24])

In [42]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs )

In [43]:
torch.as_tensor([1, 2, 3])

tensor([1, 2, 3])

In [44]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)), len(idxs)=24
[1] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on _Reshape_3747(), len(idxs)=24
[2] prune_out_channels on _Reshape_3747() => prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0), len(idxs)=24
[3] prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0) => prune_out_channels on _ElementWiseOp_3740(MulBackward0), len(idxs)=24
[4] prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0) => prune_out_c

In [45]:
for i, (dep, idx) in enumerate(group):
    if (isinstance(dep.target.module, nn.Parameter)):
        print(dep.target.name)
        print("q norm", dep.target.module is bridge.blocks[0].attn.q_norm.weight)
        print("k norm", dep.target.module is bridge.blocks[0].attn.k_norm.weight)
        print(dep.handler)
        print(idx)
        # print(dep.target.module)

UnwrappedParameter_419 (torch.Size([256]))
q norm False
k norm True
<bound method ParameterPruner.prune_out_channels of <torch_pruning.pruner.function.ParameterPruner object at 0x7c4111d11240>>
[tensor(2), tensor(6), tensor(9), tensor(258), tensor(262), tensor(265), tensor(514), tensor(518), tensor(521), tensor(770), tensor(774), tensor(777), tensor(1026), tensor(1030), tensor(1033), tensor(1282), tensor(1286), tensor(1289), tensor(1538), tensor(1542), tensor(1545), tensor(1794), tensor(1798), tensor(1801)]
UnwrappedParameter_0 (torch.Size([256]))
q norm True
k norm False
<bound method ParameterPruner.prune_out_channels of <torch_pruning.pruner.function.ParameterPruner object at 0x7c4111d11240>>
[tensor(2), tensor(6), tensor(9), tensor(258), tensor(262), tensor(265), tensor(514), tensor(518), tensor(521), tensor(770), tensor(774), tensor(777), tensor(1026), tensor(1030), tensor(1033), tensor(1282), tensor(1286), tensor(1289), tensor(1538), tensor(1542), tensor(1545), tensor(1794), tens

Why only q_norm and k_norm found? Because v_norm without .weight.

TODO: for tp all good with RMS, but for transfer_pruning .mean will work wrong!

In [46]:
bridge.get_submodule("blocks.0._original_component.self_attn._original_component.k_proj")

LinearBridge(2560 -> 512, bias=False, original_component=Linear)

### Fraction-based pruning comparison

Restart the kernel and run the model/dataset setup cells, but skip the preceding explicit-index pruning cell. This experiment samples channel indices from fractions, records the actual independently sampled RoPE coordinates for each attention layer, then compares activation and structural pruning using the same groups.

In [47]:
# Fraction-based version of the activation-vs-structural comparison.
# Run on a freshly loaded, unpruned bridge; skip the preceding explicit-index
# comparison cell after restarting the kernel.
from compare_utils import compare_tp_and_transfer_pruning

FRACTION_ATTN_LAYERS = [0, 8, 16]
FRACTION_MLP_LAYERS = [4, 12, 20]
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = [1, 2]
MLP_OUT_FRACTION = [3, 5]
FRACTION_SEED = 0

# compare_tp_and_transfer_pruning(bridge,
#                                 FRACTION_ATTN_LAYERS,
#                                 FRACTION_MLP_LAYERS,
#                                 ATTN_OUT_FRACTION,
#                                 MLP_OUT_FRACTION,
#                                 FRACTION_SEED,
#                                 evaluation_blocks,
#                                 EVAL_BATCH_SIZE
#                                 )

So we see a little divergence in more fraction sizes. But it very close.

In [48]:
import gc
import torch
import torch_pruning as tp

from lora_transfer_pruning.usecase.local_pruning import LocalPruning
from lora_transfer_pruning.usecase.prune_task_type import PruneTaskType
from experiments.utils import evaluate_language_model
from rope_resize_for_tp import make_gemma_rope_resize_pre_hook



def compare_tp_and_transfer_pruning(model_bridge, 
                                    fraction_attn_layers: list[int], 
                                    fraction_mlp_layers: list[int], 
                                    attn_out_fraction: list[int] | float, 
                                    mlp_out_fraction: list[int] | float, 
                                    seed: int, 
                                    evaluation_batches: torch.Tensor,
                                    eval_batch_size: int
                                    ):
    DEVICE = model_bridge.device
    attn_config = model_bridge.blocks[0].attn._original_component.config
    original_head_dim = int(attn_config.head_dim)
    expected_q_width = int(attn_config.num_attention_heads) * original_head_dim
    for layer in fraction_attn_layers:
        actual_q_width = model_bridge.blocks[layer].attn.q._original_component.out_features
        assert actual_q_width == expected_q_width, (
            "Fraction experiment requires a fresh, unpruned model_bridge. "
            f"Layer {layer}: q.out_features={actual_q_width}, expected={expected_q_width}. "
            "Restart the kernel, run the setup cells, skip the explicit-index pruning cell, "
            "then run this cell."
        )

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    fraction_prune_task: PruneTaskType = {
        **{
            f"blocks.{layer}.attn.q": (None, attn_out_fraction)
            for layer in fraction_attn_layers
        },
        **{
            f"blocks.{layer}.mlp.up_proj": (None, mlp_out_fraction)
            for layer in fraction_mlp_layers
        },
    }
    print("fraction prune_task:")
    for module_name, task in fraction_prune_task.items():
        print(f"  {module_name}: {task}")

    model_bridge.reset_hooks()
    fraction_local_pruning = LocalPruning(
        model_bridge,
        evaluation_batches[:1].to(DEVICE),
    )
    fraction_groups = fraction_local_pruning.get_torch_pruning_groups(
        fraction_prune_task
    )
    assert len(fraction_groups) == len(fraction_prune_task)

    # Save every root index set and the local RoPE coordinates for attention.
    fraction_removed_idxs = {}
    fraction_attn_local_idxs = {}
    for (module_name, _), pruning_group in zip(
        fraction_prune_task.items(), fraction_groups
    ):
        root_item = pruning_group[0]
        root_hybrid_idxs = root_item.idxs
        root_flat_idxs = sorted({
            int(idx) for idx in tp._helpers.to_plain_idxs(root_hybrid_idxs)
        })
        fraction_removed_idxs[module_name] = root_flat_idxs

        if ".attn.q" not in module_name:
            continue
        layer = int(module_name.split(".")[1])
        fraction_attn_local_idxs[layer] = torch.tensor(
            sorted({idx % original_head_dim for idx in root_flat_idxs}),
            dtype=torch.long,
        )

    print("removed root indices by module:")
    for module_name, removed_idxs in fraction_removed_idxs.items():
        print(
            f"  {module_name}: count={len(removed_idxs)}, idxs={removed_idxs}"
        )

    for layer, local_idxs in fraction_attn_local_idxs.items():
        paired = torch.where(
            local_idxs < original_head_dim // 2,
            local_idxs + original_head_dim // 2,
            local_idxs - original_head_dim // 2,
        )
        assert set(local_idxs.tolist()) == set(paired.tolist())
        print(
            f"layer={layer}: removed local head dims={len(local_idxs)} "
            f"({len(local_idxs) / original_head_dim:.3%}), idxs={local_idxs.tolist()}"
        )

    fraction_baseline_metrics = evaluate_language_model(
        model_bridge, evaluation_batches, batch_size=eval_batch_size
    )

    fraction_local_pruning.prepare_model_to_transfer_pruning_from_groups(fraction_groups)
    fraction_transfer_metrics = evaluate_language_model(
        model_bridge, evaluation_batches, batch_size=eval_batch_size
    )

    model_bridge.reset_hooks()
    for pruning_group in fraction_groups:
        pruning_group.prune()

    # Groups retain DependencyGraph's autograd trace.
    del fraction_groups, fraction_local_pruning, pruning_group 
    globals().pop("DG", None)
    globals().pop("group", None)
    globals().pop("group_kv", None)
    gc.collect()
    torch.cuda.empty_cache()


    #-------manual model changing for TP, cos sin hooks for Rope and saving shapes--------
    def make_fraction_compact_rope_hook(keep_idxs):
        def compact_rope_hook(activation, hook):
            return activation.index_select(-1, keep_idxs.to(activation.device))
        return compact_rope_hook

    fraction_structural_shapes = {}
    for layer, local_idxs in fraction_attn_local_idxs.items():
        removed = set(local_idxs.tolist())
        keep_idxs = torch.tensor(
            [idx for idx in range(original_head_dim) if idx not in removed],
            dtype=torch.long,
        )
        attn = model_bridge.blocks[layer].attn
        attn._original_component.head_dim = len(keep_idxs)
        attn.register_forward_pre_hook( #no need to unregister
                    make_gemma_rope_resize_pre_hook(keep_idxs),
                    with_kwargs=True,
                )
        fraction_structural_shapes[f"blocks.{layer}.attn"] = {
            "q": tuple(attn.q._original_component.weight.shape),
            "k": tuple(attn.k._original_component.weight.shape),
            "v": tuple(attn.v._original_component.weight.shape),
            "o": tuple(attn.o._original_component.weight.shape),
            "head_dim": attn._original_component.head_dim,
        }
    #----------------------------------------   

    for layer in fraction_mlp_layers:
        mlp = model_bridge.blocks[layer].mlp
        fraction_structural_shapes[f"blocks.{layer}.mlp"] = {
            "up": tuple(mlp.up_proj._original_component.weight.shape),
            "gate": tuple(mlp.gate_proj._original_component.weight.shape),
            "out": tuple(mlp.out._original_component.weight.shape),
        }

    print("fraction structural shapes:", fraction_structural_shapes)
    fraction_structural_metrics = evaluate_language_model(
        model_bridge, evaluation_batches, batch_size=eval_batch_size
    )

    fraction_comparison = {
        "baseline": fraction_baseline_metrics,
        "transfer_activation": fraction_transfer_metrics,
        "torch_pruning_structural": fraction_structural_metrics,
        "structural_minus_transfer": {
            key: fraction_structural_metrics[key] - fraction_transfer_metrics[key]
            for key in ("loss", "perplexity")
        },
    }
    return fraction_comparison

In [49]:
bridge.blocks[0].attn

GeneralizedComponent(
  (hook_in): HookPoint(name='blocks.0.attn.hook_in')
  (hook_out): HookPoint(name='blocks.0.attn.hook_out')
  (_original_component): Gemma4TextAttention(
    (q_norm): GeneralizedComponent(
      (hook_in): HookPoint(name='blocks.0.attn.q_norm.hook_in')
      (hook_out): HookPoint(name='blocks.0.attn.q_norm.hook_out')
      (_original_component): Gemma4RMSNorm()
    )
    (k_norm): GeneralizedComponent(
      (hook_in): HookPoint(name='blocks.0.attn.k_norm.hook_in')
      (hook_out): HookPoint(name='blocks.0.attn.k_norm.hook_out')
      (_original_component): Gemma4RMSNorm()
    )
    (v_norm): GeneralizedComponent(
      (hook_in): HookPoint(name='blocks.0.attn.v_norm.hook_in')
      (hook_out): HookPoint(name='blocks.0.attn.v_norm.hook_out')
      (_original_component): Gemma4RMSNorm()
    )
    (k_proj): LinearBridge(2560 -> 512, bias=False, original_component=Linear)
    (q_proj): LinearBridge(2560 -> 2048, bias=False, original_component=Linear)
    (v_proj): 

In [50]:
compare_tp_and_transfer_pruning(bridge,
                                FRACTION_ATTN_LAYERS,
                                FRACTION_MLP_LAYERS,
                                ATTN_OUT_FRACTION,
                                MLP_OUT_FRACTION,
                                FRACTION_SEED,
                                evaluation_blocks,
                                EVAL_BATCH_SIZE
                                )

fraction prune_task:
  blocks.0.attn.q: (None, [1, 2])
  blocks.8.attn.q: (None, [1, 2])
  blocks.16.attn.q: (None, [1, 2])
  blocks.4.mlp.up_proj: (None, [3, 5])
  blocks.12.mlp.up_proj: (None, [3, 5])
  blocks.20.mlp.up_proj: (None, [3, 5])


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.vision_tower._original_component.encoder.layers.9.post_attention_layernorm.weight', 'model.vision_tower._original_component.encoder.layers.13.self_attn.q_norm.weight', 'model.audio_tower.layers.6.feed_forward2.pre_layer_norm.weight', 'model.audio_tower.layers.6.lconv1d.linear_start.linear.weight', 'model.audio_tower.layers.9.feed_forward1.post_layer_norm.weight', 'model.language_model.layers.2._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.8._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.13._original_component.pre_feedforward_layernorm._original_component.weight', 'model.language_model.layers.22._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.28._original_compo

removed root indices by module:
  blocks.0.attn.q: count=32, idxs=[1, 2, 129, 130, 257, 258, 385, 386, 513, 514, 641, 642, 769, 770, 897, 898, 1025, 1026, 1153, 1154, 1281, 1282, 1409, 1410, 1537, 1538, 1665, 1666, 1793, 1794, 1921, 1922]
  blocks.8.attn.q: count=32, idxs=[1, 2, 129, 130, 257, 258, 385, 386, 513, 514, 641, 642, 769, 770, 897, 898, 1025, 1026, 1153, 1154, 1281, 1282, 1409, 1410, 1537, 1538, 1665, 1666, 1793, 1794, 1921, 1922]
  blocks.16.attn.q: count=32, idxs=[1, 2, 129, 130, 257, 258, 385, 386, 513, 514, 641, 642, 769, 770, 897, 898, 1025, 1026, 1153, 1154, 1281, 1282, 1409, 1410, 1537, 1538, 1665, 1666, 1793, 1794, 1921, 1922]
  blocks.4.mlp.up_proj: count=2, idxs=[3, 5]
  blocks.12.mlp.up_proj: count=2, idxs=[3, 5]
  blocks.20.mlp.up_proj: count=2, idxs=[3, 5]
layer=0: removed local head dims=4 (1.562%), idxs=[1, 2, 129, 130]
layer=8: removed local head dims=4 (1.562%), idxs=[1, 2, 129, 130]
layer=16: removed local head dims=4 (1.562%), idxs=[1, 2, 129, 130]
fractio

{'baseline': {'loss': 8.498046875, 'perplexity': 4905.17919921875},
 'transfer_activation': {'loss': 8.5166015625, 'perplexity': 4997.04296875},
 'torch_pruning_structural': {'loss': 8.5068359375,
  'perplexity': 4948.48095703125},
 'structural_minus_transfer': {'loss': -0.009765625,
  'perplexity': -48.56201171875}}

TODO: check layer by layer + check norms